In [34]:
import numpy as np
import pandas as pd
import re
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Lambda,Embedding

In [35]:
text = """
The speed of transmission is an important point of difference between the two viruses. Influenza has a shorter median incubation period (the time from infection to appearance of symptoms) and a shorter serial interval (the time between successive cases) than COVID-19 virus. The serial interval for COVID-19 virus is estimated to be 5-6 days, while for influenza virus, the serial interval is 3 days. This means that influenza can spread faster than COVID-19.

Further, transmission in the first 3-5 days of illness, or potentially pre-symptomatic transmission –transmission of the virus before the appearance of symptoms – is a major driver of transmission for influenza. In contrast, while we are learning that there are people who can shed COVID-19 virus 24-48 hours prior to symptom onset, at present, this does not appear to be a major driver of transmission.

The reproductive number – the number of secondary infections generated from one infected individual – is understood to be between 2 and 2.5 for COVID-19 virus, higher than for influenza. However, estimates for both COVID-19 and influenza viruses are very context and time-specific, making direct comparisons more difficult.
"""

In [36]:
text = re.sub(r'[^a-z\s]', '',text.lower())
words = text.split()

In [37]:
vocab = list(set(words))
word_to_index = {w:i for i,w in enumerate(vocab)}
index_to_word = {i:w for w,i in word_to_index.items()}
vocab_size = len(vocab)

In [38]:
print("Vocabulary : ",word_to_index)

Vocabulary :  {'while': 0, 'before': 1, 'that': 2, 'presymptomatic': 3, 'individual': 4, 'period': 5, 'in': 6, 'shed': 7, 'driver': 8, 'time': 9, 'spread': 10, 'covid': 11, 'successive': 12, 'estimated': 13, 'median': 14, 'to': 15, 'difficult': 16, 'who': 17, 'onset': 18, 'hours': 19, 'a': 20, 'or': 21, 'this': 22, 'higher': 23, 'first': 24, 'infection': 25, 'more': 26, 'faster': 27, 'however': 28, 'making': 29, 'people': 30, 'for': 31, 'can': 32, 'number': 33, 'generated': 34, 'potentially': 35, 'an': 36, 'means': 37, 'shorter': 38, 'viruses': 39, 'context': 40, 'direct': 41, 'symptom': 42, 'interval': 43, 'two': 44, 'estimates': 45, 'serial': 46, 'does': 47, 'we': 48, 'one': 49, 'not': 50, 'from': 51, 'infections': 52, 'and': 53, 'transmission': 54, 'the': 55, 'is': 56, 'than': 57, 'contrast': 58, 'appear': 59, 'learning': 60, 'at': 61, 'are': 62, 'appearance': 63, 'influenza': 64, 'virus': 65, 'between': 66, 'present': 67, 'illness': 68, 'reproductive': 69, 'point': 70, 'difference'

In [39]:
x,y = [],[]
window = 2

In [40]:
for i in range(window, len(words)-window):
    context = [word_to_index[w] for w in
               words[i-2:i] + words[i+1:i+3]]
    target = word_to_index[words[i]]
    x.append(context)
    y.append(target)

In [41]:
x = np.array(x)
y=np.array(y)

In [42]:
model = Sequential([
    Embedding(92,10),
    Lambda(lambda x: tf.reduce_mean(x,axis=1)),
    Dense(92,activation = 'softmax')
])

In [44]:
model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)

In [45]:
history = model.fit(
    x,y,
    epochs = 100,
    verbose = 0
)

In [46]:
embeddings = model.layers[0].get_weights()[0]

In [47]:
print("\nLearned word embeddings:")
for word,index in word_to_index.items():
    print(word,embeddings[index])


Learned word embeddings:
while [-0.39432946 -0.18440922 -0.6754928   0.04399883 -0.22885656  0.5469359
  0.12364344 -0.00326107  0.3357571  -0.33583438]
before [ 0.15496823  0.5713712  -0.45057943 -0.0772442  -0.6171218   0.20707074
 -0.391972    0.07089525 -0.24608035 -0.587541  ]
that [-0.474      -0.44746587 -0.7470372   0.06717525  0.01446292  0.7100969
  0.44308087  0.34110576  0.60382736  0.11166158]
presymptomatic [ 0.4373109  -0.13430192  0.00260548 -0.28351417 -0.5487129   0.25607145
 -0.25371027  0.15099303 -0.48817292  0.25814015]
individual [ 0.18146223  0.20275442  0.37600863  0.3375552  -0.06605688 -0.14558566
 -0.00481298 -0.5104183  -0.11990856  0.02648536]
period [ 0.37425086  0.15126157 -0.19066377 -0.40474412  0.13490874  0.20198521
 -0.23608033  0.31205723  0.11377451 -0.22568072]
in [ 0.33004662  0.09335591 -0.6963966  -0.26012254 -0.62112683  0.7024771
 -0.29563323 -0.15861644  0.10704415  0.12453423]
shed [-0.5030796   0.56163496 -0.30358005  0.5380121  -0.09626

In [48]:
context = ["speed","of","is","an"]

In [51]:
test = np.array([[word_to_index[w] for w in context]])

In [56]:
print("\nTraining loss = ",history.history['loss'][-1]*100)


Training loss =  333.9836597442627


In [52]:
prediction = model.predict(test,verbose = 0)

In [55]:
print("\nPredicted target word:",
      index_to_word[np.argmax(prediction)])


Predicted target word: the
